# CodeMix-T — Phase 2: Custom Transformer Architecture

This notebook implements the full **CodeMix-T** architecture from scratch in PyTorch.

### Architecture Overview
```
Input (code-mixed tokens)
        │
        ▼
┌─────────────────────────────────┐
│  Token Embedding                │
│+ Positional Embedding           │
│+ Language-ID Embedding  ◄── NOVEL│
└────────────┬────────────────────┘
             │
        ┌────▼─────┐
        │ Encoder  │  6 layers
        │ Stack    │  8 heads
        └────┬─────┘
             │ (encoder memory)
        ┌────▼─────┐
        │ Decoder  │  6 layers
        │ Stack    │  cross-attention
        └────┬─────┘
             │
        ┌────▼─────┐
        │ Linear   │  project to vocab
        │ + Softmax│
        └──────────┘
             │
     English translation
```

### Components Built
1. Config dataclass
2. Token + Positional + Language-ID Embeddings
3. Multi-Head Attention
4. Position-wise Feed Forward
5. Encoder Layer + Encoder Stack
6. Decoder Layer + Decoder Stack
7. Full CodeMixT Model
8. Beam Search Decoder
9. Chatbot Interface
10. Model summary + sanity checks

## Cell 1 — Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from dataclasses import dataclass
from typing import Optional
import copy

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — Model Configuration

In [ ]:
@dataclass
class CodeMixTConfig:
    """
    Central config for CodeMix-T architecture.
    All hyperparameters in one place — easy to swap between small/base.
    """
    # Vocabulary
    vocab_size: int     = 16000   # Must match your trained tokenizer
    pad_id: int         = 0
    bos_id: int         = 2
    eos_id: int         = 3

    # Language ID (novel component)
    num_languages: int  = 4       # EN=0, HI=1, TA=2, UNK=3
    lang_embed_dim: int = 64      # Dimension of language-ID embedding

    # Model dimensions
    d_model: int        = 512     # Main model dimension
    d_ff: int           = 2048    # Feed-forward hidden dimension
    num_heads: int      = 8       # Attention heads
    num_enc_layers: int = 6       # Encoder layers
    num_dec_layers: int = 6       # Decoder layers
    max_seq_len: int    = 128     # Max sequence length
    dropout: float      = 0.1

    # Inference
    beam_size: int      = 4
    max_gen_len: int    = 128

    def __post_init__(self):
        assert self.d_model % self.num_heads == 0, \
            f'd_model ({self.d_model}) must be divisible by num_heads ({self.num_heads})'
        self.d_k = self.d_model // self.num_heads  # Key/query dimension per head


# Two preset configs
CONFIG_SMALL = CodeMixTConfig(
    d_model=256, d_ff=1024, num_heads=4,
    num_enc_layers=4, num_dec_layers=4,
    lang_embed_dim=32
)

CONFIG_BASE = CodeMixTConfig()  # Default values above (~80M params)

# Use BASE for training, SMALL for quick debugging
cfg = CONFIG_BASE
print('Config loaded:')
print(f'  d_model={cfg.d_model}, heads={cfg.num_heads}, d_k={cfg.d_k}')
print(f'  enc_layers={cfg.num_enc_layers}, dec_layers={cfg.num_dec_layers}')
print(f'  vocab_size={cfg.vocab_size}, lang_embed_dim={cfg.lang_embed_dim}')

## Cell 3 — Input Embeddings (Token + Positional + Language-ID)

This is the **novel contribution**: three embeddings summed together.
Standard Transformers use Token + Positional only.
We add Language-ID embeddings so the model explicitly knows which language each token belongs to.

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding (Vaswani et al. 2017).
    Fixed (not learned) — encodes token position in sequence.
    
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model: int, max_len: int, dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Build PE matrix [max_len, d_model]
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)  # Even dims
        pe[:, 1::2] = torch.cos(position * div_term)  # Odd dims
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]

        # Register as buffer (not a parameter, but saved with model)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, seq_len, d_model]
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class CodeMixEmbedding(nn.Module):
    """
    Three-component embedding for CodeMix-T.

    Output = TokenEmbed(token_ids)        [batch, seq, d_model]
           + PosEncode(position)          [batch, seq, d_model]
           + LangIDEmbed(lang_ids) → proj [batch, seq, d_model]

    The Language-ID embedding is projected from lang_embed_dim → d_model
    via a learned linear layer before addition.
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()

        # 1. Standard token embedding
        self.token_embed = nn.Embedding(
            cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id
        )

        # 2. Sinusoidal positional encoding
        self.pos_encoding = PositionalEncoding(
            cfg.d_model, cfg.max_seq_len, cfg.dropout
        )

        # 3. Language-ID embedding (NOVEL COMPONENT)
        #    Maps language tag (0,1,2,3) → lang_embed_dim
        #    Then projects to d_model for addition
        self.lang_embed = nn.Embedding(cfg.num_languages, cfg.lang_embed_dim)
        self.lang_proj  = nn.Linear(cfg.lang_embed_dim, cfg.d_model, bias=False)

        self.dropout = nn.Dropout(cfg.dropout)

        # Scale token embeddings (standard practice from Vaswani et al.)
        self.scale = math.sqrt(cfg.d_model)

    def forward(
        self,
        token_ids: torch.Tensor,   # [batch, seq_len]
        lang_ids: torch.Tensor     # [batch, seq_len]  — Language ID per token
    ) -> torch.Tensor:
        # Token embedding scaled
        tok = self.token_embed(token_ids) * self.scale  # [B, S, d_model]

        # Language-ID embedding projected to d_model
        lang = self.lang_proj(self.lang_embed(lang_ids))  # [B, S, d_model]

        # Sum all three (positional encoding adds inside pos_encoding)
        x = tok + lang                    # Token + Language
        x = self.pos_encoding(x)          # + Positional
        return x                          # [B, S, d_model]


print('Embedding classes defined.')

# Quick shape test
_emb = CodeMixEmbedding(cfg)
_tok = torch.randint(0, cfg.vocab_size, (2, 10))
_lid = torch.randint(0, cfg.num_languages, (2, 10))
_out = _emb(_tok, _lid)
print(f'Embedding output shape: {_out.shape}  (expected [2, 10, {cfg.d_model}])')
del _emb, _tok, _lid, _out

## Cell 4 — Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Scaled Dot-Product Attention.

    Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V

    Used for:
      - Encoder self-attention
      - Decoder masked self-attention  (causal mask)
      - Decoder cross-attention        (attends to encoder output)
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.num_heads = cfg.num_heads
        self.d_k       = cfg.d_k
        self.d_model   = cfg.d_model

        # Projection matrices for Q, K, V, and output
        self.W_q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

        self.dropout = nn.Dropout(cfg.dropout)
        self.scale   = math.sqrt(self.d_k)

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[B, S, d_model] → [B, heads, S, d_k]"""
        B, S, _ = x.shape
        x = x.view(B, S, self.num_heads, self.d_k)
        return x.transpose(1, 2)  # [B, heads, S, d_k]

    def merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[B, heads, S, d_k] → [B, S, d_model]"""
        B, _, S, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, S, self.d_model)

    def forward(
        self,
        query: torch.Tensor,                # [B, S_q, d_model]
        key:   torch.Tensor,                # [B, S_k, d_model]
        value: torch.Tensor,                # [B, S_k, d_model]
        mask:  Optional[torch.Tensor]=None  # Attention mask
    ) -> torch.Tensor:

        Q = self.split_heads(self.W_q(query))  # [B, h, S_q, d_k]
        K = self.split_heads(self.W_k(key))    # [B, h, S_k, d_k]
        V = self.split_heads(self.W_v(value))  # [B, h, S_k, d_k]

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # [B, h, S_q, S_k]

        if mask is not None:
            # Fill masked positions with -inf so softmax → ~0
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = self.dropout(F.softmax(scores, dim=-1))  # [B, h, S_q, S_k]

        # Weighted sum of values
        context = torch.matmul(attn_weights, V)   # [B, h, S_q, d_k]
        context = self.merge_heads(context)        # [B, S_q, d_model]

        return self.W_o(context)                   # [B, S_q, d_model]


print('MultiHeadAttention defined.')

# Shape test
_mha = MultiHeadAttention(cfg)
_q = torch.randn(2, 10, cfg.d_model)
_k = torch.randn(2, 15, cfg.d_model)
_v = torch.randn(2, 15, cfg.d_model)
_out = _mha(_q, _k, _v)
print(f'MHA output shape: {_out.shape}  (expected [2, 10, {cfg.d_model}])')
del _mha, _q, _k, _v, _out

## Cell 5 — Position-wise Feed Forward Network

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """
    FFN(x) = max(0, xW1 + b1)W2 + b2
    Applied independently to each position.
    d_model → d_ff → d_model
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.linear1 = nn.Linear(cfg.d_model, cfg.d_ff)
        self.linear2 = nn.Linear(cfg.d_ff, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, S, d_model]
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))
        # Note: using GELU instead of ReLU — smoother gradients, used in BERT/GPT


print('PositionwiseFeedForward defined.')

## Cell 6 — Encoder Layer

In [ ]:
class EncoderLayer(nn.Module):
    """
    Single Transformer encoder layer.

    Using Pre-LN (Layer Norm before sublayer) for more stable training:
        x = x + SelfAttn(LayerNorm(x))
        x = x + FFN(LayerNorm(x))

    (Post-LN = original Vaswani paper; Pre-LN = used in GPT-2, more stable)
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.self_attn = MultiHeadAttention(cfg)
        self.ffn       = PositionwiseFeedForward(cfg)
        self.norm1     = nn.LayerNorm(cfg.d_model)
        self.norm2     = nn.LayerNorm(cfg.d_model)
        self.dropout   = nn.Dropout(cfg.dropout)

    def forward(
        self,
        x:    torch.Tensor,                  # [B, S, d_model]
        mask: Optional[torch.Tensor] = None  # Padding mask
    ) -> torch.Tensor:
        # Pre-LN self-attention + residual
        residual = x
        x = self.norm1(x)
        x = residual + self.dropout(self.self_attn(x, x, x, mask))

        # Pre-LN FFN + residual
        residual = x
        x = self.norm2(x)
        x = residual + self.dropout(self.ffn(x))

        return x  # [B, S, d_model]


class Encoder(nn.Module):
    """
    Full encoder: Embedding → N × EncoderLayer → LayerNorm
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.embedding = CodeMixEmbedding(cfg)
        self.layers    = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.num_enc_layers)])
        self.norm      = nn.LayerNorm(cfg.d_model)

    def forward(
        self,
        token_ids: torch.Tensor,  # [B, src_len]
        lang_ids:  torch.Tensor,  # [B, src_len]
        src_mask:  Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        x = self.embedding(token_ids, lang_ids)  # [B, src_len, d_model]
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)  # [B, src_len, d_model]


print('EncoderLayer and Encoder defined.')

# Shape test
_enc = Encoder(cfg)
_tok = torch.randint(0, cfg.vocab_size, (2, 12))
_lid = torch.randint(0, cfg.num_languages, (2, 12))
_out = _enc(_tok, _lid)
print(f'Encoder output shape: {_out.shape}  (expected [2, 12, {cfg.d_model}])')
del _enc, _tok, _lid, _out

## Cell 7 — Decoder Layer

In [ ]:
class DecoderLayer(nn.Module):
    """
    Single Transformer decoder layer — three sublayers:
      1. Masked self-attention  (causal: can't see future tokens)
      2. Cross-attention        (attends to encoder memory)
      3. Feed-forward network

    Pre-LN variant for training stability.
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.self_attn  = MultiHeadAttention(cfg)  # Masked self-attn
        self.cross_attn = MultiHeadAttention(cfg)  # Cross-attn over encoder
        self.ffn        = PositionwiseFeedForward(cfg)
        self.norm1      = nn.LayerNorm(cfg.d_model)
        self.norm2      = nn.LayerNorm(cfg.d_model)
        self.norm3      = nn.LayerNorm(cfg.d_model)
        self.dropout    = nn.Dropout(cfg.dropout)

    def forward(
        self,
        x:          torch.Tensor,                   # [B, tgt_len, d_model]
        enc_output: torch.Tensor,                   # [B, src_len, d_model]
        tgt_mask:   Optional[torch.Tensor] = None,  # Causal mask
        src_mask:   Optional[torch.Tensor] = None   # Padding mask
    ) -> torch.Tensor:

        # 1. Masked self-attention
        residual = x
        x = self.norm1(x)
        x = residual + self.dropout(self.self_attn(x, x, x, tgt_mask))

        # 2. Cross-attention (Q from decoder, K/V from encoder)
        residual = x
        x = self.norm2(x)
        x = residual + self.dropout(self.cross_attn(x, enc_output, enc_output, src_mask))

        # 3. Feed-forward
        residual = x
        x = self.norm3(x)
        x = residual + self.dropout(self.ffn(x))

        return x  # [B, tgt_len, d_model]


class Decoder(nn.Module):
    """
    Full decoder: Embedding → N × DecoderLayer → LayerNorm
    Note: Decoder uses standard token embedding only (no lang-ID needed
    for the English output side — English is always LANG_EN).
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        # Standard embedding for target (English) — no language-ID needed
        self.token_embed  = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.pos_encoding = PositionalEncoding(cfg.d_model, cfg.max_seq_len, cfg.dropout)
        self.layers       = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.num_dec_layers)])
        self.norm         = nn.LayerNorm(cfg.d_model)
        self.scale        = math.sqrt(cfg.d_model)

    def forward(
        self,
        tgt_ids:    torch.Tensor,                   # [B, tgt_len]
        enc_output: torch.Tensor,                   # [B, src_len, d_model]
        tgt_mask:   Optional[torch.Tensor] = None,
        src_mask:   Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        x = self.token_embed(tgt_ids) * self.scale  # [B, tgt_len, d_model]
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, enc_output, tgt_mask, src_mask)
        return self.norm(x)  # [B, tgt_len, d_model]


print('DecoderLayer and Decoder defined.')

## Cell 8 — Mask Utilities

In [ ]:
def make_src_mask(src: torch.Tensor, pad_id: int) -> torch.Tensor:
    """
    Source padding mask — hides PAD tokens from attention.
    Returns: [B, 1, 1, src_len]  (broadcast over heads and query positions)
    """
    return (src != pad_id).unsqueeze(1).unsqueeze(2)  # [B, 1, 1, src_len]


def make_tgt_mask(tgt: torch.Tensor, pad_id: int) -> torch.Tensor:
    """
    Target causal mask — combines:
      1. Causal (lower-triangular) mask: token i can't attend to j > i
      2. Padding mask: ignore PAD tokens

    Returns: [B, 1, tgt_len, tgt_len]
    """
    B, T = tgt.shape

    # Padding mask: [B, 1, 1, T]
    pad_mask = (tgt != pad_id).unsqueeze(1).unsqueeze(2)

    # Causal mask: [1, 1, T, T]  — lower triangular
    causal_mask = torch.tril(torch.ones(T, T, device=tgt.device)).bool()
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

    # Combine: both conditions must be True
    return pad_mask & causal_mask  # [B, 1, T, T]


# Test masks
_src = torch.tensor([[5, 10, 3, 0, 0]])      # Last 2 are PAD
_tgt = torch.tensor([[2, 5, 10, 3]])          # <bos> token token <eos>

src_m = make_src_mask(_src, pad_id=0)
tgt_m = make_tgt_mask(_tgt, pad_id=0)

print(f'src_mask shape: {src_m.shape}')  # [1, 1, 1, 5]
print(f'src_mask:       {src_m}')
print(f'tgt_mask shape: {tgt_m.shape}')  # [1, 1, 4, 4]
print(f'tgt_mask (causal):\n{tgt_m[0,0]}')
del _src, _tgt, src_m, tgt_m

## Cell 9 — Full CodeMix-T Model

In [ ]:
class CodeMixT(nn.Module):
    """
    CodeMix-T: Full encoder-decoder Transformer for code-mixed translation.

    Novel features:
      - Language-ID-Aware Embeddings in the encoder
      - Trained from scratch on Tanglish + Hinglish corpora
      - Custom BPE tokenizer covering mixed-script tokens
    """
    def __init__(self, cfg: CodeMixTConfig):
        super().__init__()
        self.cfg     = cfg
        self.encoder = Encoder(cfg)
        self.decoder = Decoder(cfg)

        # Final projection: d_model → vocab_size
        self.output_projection = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Weight tying: share decoder embedding weights with output projection
        # (standard practice — reduces parameters, improves performance)
        self.output_projection.weight = self.decoder.token_embed.weight

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Xavier uniform init for linear layers, normal for embeddings."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0, std=0.02)
                if module.padding_idx is not None:
                    module.weight.data[module.padding_idx].zero_()

    def encode(
        self,
        src_ids:  torch.Tensor,  # [B, src_len]
        lang_ids: torch.Tensor,  # [B, src_len]
        src_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """Encode source sequence → encoder memory."""
        return self.encoder(src_ids, lang_ids, src_mask)

    def decode(
        self,
        tgt_ids:    torch.Tensor,
        enc_output: torch.Tensor,
        tgt_mask:   Optional[torch.Tensor] = None,
        src_mask:   Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """Decode target sequence given encoder memory."""
        dec_out = self.decoder(tgt_ids, enc_output, tgt_mask, src_mask)
        return self.output_projection(dec_out)  # [B, tgt_len, vocab_size]

    def forward(
        self,
        src_ids:  torch.Tensor,  # [B, src_len]
        lang_ids: torch.Tensor,  # [B, src_len]
        tgt_ids:  torch.Tensor,  # [B, tgt_len]
    ) -> torch.Tensor:
        """
        Full forward pass for training.
        Returns logits: [B, tgt_len, vocab_size]
        """
        src_mask = make_src_mask(src_ids, self.cfg.pad_id).to(src_ids.device)
        tgt_mask = make_tgt_mask(tgt_ids, self.cfg.pad_id).to(tgt_ids.device)

        enc_output = self.encode(src_ids, lang_ids, src_mask)
        logits     = self.decode(tgt_ids, enc_output, tgt_mask, src_mask)
        return logits  # [B, tgt_len, vocab_size]

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Instantiate and check
model = CodeMixT(cfg).to(device)
n_params = model.count_parameters()
print(f'CodeMix-T instantiated successfully.')
print(f'Total trainable parameters: {n_params:,}  ({n_params/1e6:.1f}M)')

# Forward pass test
B, SRC_LEN, TGT_LEN = 2, 20, 15
_src  = torch.randint(1, cfg.vocab_size, (B, SRC_LEN)).to(device)
_lang = torch.randint(0, cfg.num_languages, (B, SRC_LEN)).to(device)
_tgt  = torch.randint(1, cfg.vocab_size, (B, TGT_LEN)).to(device)

with torch.no_grad():
    logits = model(_src, _lang, _tgt)

print(f'Forward pass output shape: {logits.shape}  (expected [{B}, {TGT_LEN}, {cfg.vocab_size}])')
del _src, _lang, _tgt

## Cell 10 — Beam Search Decoder

In [ ]:
class BeamSearchDecoder:
    """
    Beam search for inference.
    Keeps top-k candidate sequences at each step.
    Significantly better than greedy decoding for translation.
    """
    def __init__(self, model: CodeMixT, cfg: CodeMixTConfig):
        self.model    = model
        self.cfg      = cfg
        self.beam_size = cfg.beam_size

    @torch.no_grad()
    def decode(
        self,
        src_ids:  torch.Tensor,  # [1, src_len]  — single sentence
        lang_ids: torch.Tensor,  # [1, src_len]
        max_len:  int = None
    ) -> list:
        """
        Run beam search. Returns list of token IDs (best sequence).
        """
        if max_len is None:
            max_len = self.cfg.max_gen_len

        self.model.eval()
        device = src_ids.device

        # Encode source once
        src_mask   = make_src_mask(src_ids, self.cfg.pad_id).to(device)
        enc_output = self.model.encode(src_ids, lang_ids, src_mask)  # [1, src_len, d_model]

        # Expand encoder output for beam_size candidates
        enc_output = enc_output.repeat(self.beam_size, 1, 1)  # [beam, src_len, d_model]
        src_mask   = src_mask.repeat(self.beam_size, 1, 1, 1)

        # Initialize beams: (score, token_sequence)
        beams = [(0.0, [self.cfg.bos_id])]
        completed = []

        for step in range(max_len):
            candidates = []

            # Pad all beams to same length for batched decoding
            beam_seqs = [seq for _, seq in beams]
            max_beam_len = max(len(s) for s in beam_seqs)
            padded = [
                s + [self.cfg.pad_id] * (max_beam_len - len(s))
                for s in beam_seqs
            ]
            tgt_tensor = torch.tensor(padded, device=device)  # [beam, step+1]
            tgt_mask   = make_tgt_mask(tgt_tensor, self.cfg.pad_id).to(device)

            # Forward through decoder
            logits = self.model.decode(
                tgt_tensor,
                enc_output[:len(beams)],
                tgt_mask,
                src_mask[:len(beams)]
            )  # [beam, step+1, vocab]

            # Get log-probs for last token position
            log_probs = F.log_softmax(logits[:, -1, :], dim=-1)  # [beam, vocab]

            for i, (score, seq) in enumerate(beams):
                if seq[-1] == self.cfg.eos_id:
                    completed.append((score, seq))
                    continue

                # Top-k next tokens for this beam
                topk_probs, topk_ids = log_probs[i].topk(self.beam_size)
                for prob, tok_id in zip(topk_probs.tolist(), topk_ids.tolist()):
                    candidates.append((score + prob, seq + [tok_id]))

            if not candidates:
                break

            # Keep top beam_size candidates
            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = candidates[:self.beam_size]

            # Early exit if all beams ended
            if all(seq[-1] == self.cfg.eos_id for _, seq in beams):
                completed.extend(beams)
                break

        # Pick best completed sequence, or best beam
        all_seqs = completed if completed else beams
        all_seqs.sort(key=lambda x: x[0] / len(x[1]), reverse=True)  # Length-normalized
        best_seq = all_seqs[0][1]

        # Strip BOS and EOS
        if best_seq and best_seq[0] == self.cfg.bos_id:
            best_seq = best_seq[1:]
        if best_seq and best_seq[-1] == self.cfg.eos_id:
            best_seq = best_seq[:-1]

        return best_seq


beam_decoder = BeamSearchDecoder(model, cfg)
print('BeamSearchDecoder defined.')

## Cell 11 — Chatbot Interface

In [ ]:
import sentencepiece as spm
from google.colab import drive

# Language ID constants (must match Phase 1)
LANG_EN, LANG_HI, LANG_TA, LANG_UNK = 0, 1, 2, 3

HINDI_ROMANIZED_KEYWORDS = {
    'hai', 'hain', 'tha', 'thi', 'the', 'main', 'mein', 'nahi', 'nahin',
    'kya', 'koi', 'aur', 'lekin', 'par', 'toh', 'bhi', 'abhi', 'yahan',
    'wahan', 'kab', 'kyun', 'kaisa', 'kaisi', 'bahut', 'accha', 'theek',
    'kal', 'aaj', 'subah', 'raat', 'gaya', 'aya', 'dekh', 'baat', 'karo',
    'hoga', 'hogi', 'karke', 'leke', 'jaake', 'rehna', 'matlab', 'samajh',
    'pata', 'paise', 'kaam', 'log', 'dost', 'yaar', 'bhai'
}
TAMIL_ROMANIZED_KEYWORDS = {
    'naan', 'nee', 'avan', 'aval', 'avanga', 'romba', 'sollu', 'solla',
    'paar', 'paaru', 'vaa', 'po', 'enna', 'epdi', 'eppova', 'konjam',
    'koncham', 'theriyum', 'theriyala', 'irukku', 'irukken', 'pannuven',
    'pannrom', 'seri', 'illa', 'illai', 'ama', 'aama', 'inge', 'anga',
    'yaar', 'yaaru', 'enga', 'venum', 'vendam', 'paavam', 'sappa', 'super'
}


class CodeMixTChatbot:
    """
    Chatbot wrapper around CodeMix-T.
    Handles: tokenization → language tagging → inference → detokenization.
    """
    def __init__(self, model: CodeMixT, tokenizer_path: str, cfg: CodeMixTConfig):
        self.model = model
        self.cfg   = cfg
        self.beam_decoder = BeamSearchDecoder(model, cfg)

        self.sp = spm.SentencePieceProcessor()
        self.sp.load(tokenizer_path)
        print(f'Tokenizer loaded: vocab_size={self.sp.get_piece_size()}')

    def _get_token_lang_id(self, token: str) -> int:
        token_lower = token.lower().strip()
        if any('\u0900' <= c <= '\u097F' for c in token): return LANG_HI
        if any('\u0B80' <= c <= '\u0BFF' for c in token): return LANG_TA
        if token_lower in HINDI_ROMANIZED_KEYWORDS: return LANG_HI
        if token_lower in TAMIL_ROMANIZED_KEYWORDS: return LANG_TA
        return LANG_EN

    def _prepare_input(self, text: str):
        """Tokenize input and generate language ID sequence."""
        # Tokenize with SentencePiece
        pieces = self.sp.encode_as_pieces(text)
        token_ids = [self.cfg.bos_id] + self.sp.encode_as_ids(text) + [self.cfg.eos_id]

        # Assign language ID to each token
        # BOS and EOS get LANG_EN (neutral)
        lang_ids = [LANG_EN]
        for piece in pieces:
            # Clean SentencePiece prefix (▁ symbol)
            clean = piece.replace('▁', '').strip()
            lang_ids.append(self._get_token_lang_id(clean))
        lang_ids.append(LANG_EN)  # EOS

        # Truncate to max_seq_len
        token_ids = token_ids[:self.cfg.max_seq_len]
        lang_ids  = lang_ids[:self.cfg.max_seq_len]

        src_tensor  = torch.tensor([token_ids], device=device)
        lang_tensor = torch.tensor([lang_ids],  device=device)
        return src_tensor, lang_tensor

    def translate(self, text: str) -> str:
        """Translate a single code-mixed sentence to English."""
        self.model.eval()
        src_tensor, lang_tensor = self._prepare_input(text)
        output_ids = self.beam_decoder.decode(src_tensor, lang_tensor)
        return self.sp.decode_ids(output_ids)

    def chat(self):
        """Interactive chat loop in Colab."""
        print('=' * 55)
        print('  CodeMix-T Chatbot')
        print('  Type Tanglish or Hinglish to translate to English.')
        print('  Type "quit" to exit.')
        print('=' * 55)
        while True:
            user_input = input('You: ').strip()
            if user_input.lower() in ('quit', 'exit', 'q'):
                print('Goodbye!')
                break
            if not user_input:
                continue
            translation = self.translate(user_input)
            print(f'Bot: {translation}\n')


print('CodeMixTChatbot defined.')
print('NOTE: Run chatbot AFTER training (Phase 3). This is the inference interface.')

## Cell 12 — Model Summary

In [ ]:
def model_summary(model: CodeMixT):
    print('=' * 60)
    print('  CODEMIX-T MODEL SUMMARY')
    print('=' * 60)

    components = {
        'Encoder Embedding (Token+LangID+Pos)': model.encoder.embedding,
        'Encoder Layers':                       model.encoder.layers,
        'Decoder Embedding':                    model.decoder.token_embed,
        'Decoder Layers':                       model.decoder.layers,
        'Output Projection':                    model.output_projection,
    }

    total = 0
    for name, module in components.items():
        params = sum(p.numel() for p in module.parameters() if p.requires_grad)
        total += params
        print(f'  {name:<42} {params:>12,}')

    print('-' * 60)
    print(f'  {"Total (trainable)":<42} {model.count_parameters():>12,}')
    print(f'  {"Approx size (fp32)":<42} {model.count_parameters()*4/1e6:>11.1f}MB')
    print('=' * 60)

    print('\nConfig:')
    c = model.cfg
    print(f'  d_model={c.d_model}, d_ff={c.d_ff}, heads={c.num_heads}')
    print(f'  enc_layers={c.num_enc_layers}, dec_layers={c.num_dec_layers}')
    print(f'  vocab={c.vocab_size}, max_seq={c.max_seq_len}')
    print(f'  lang_embed_dim={c.lang_embed_dim}, num_languages={c.num_languages}')
    print(f'  dropout={c.dropout}, beam_size={c.beam_size}')


model_summary(model)

## Cell 13 — Architecture Sanity Checks

In [ ]:
print('Running architecture sanity checks...\n')

model.eval()
B = 4
SRC = 25
TGT = 18

# Simulate a real batch
src_ids  = torch.randint(4, cfg.vocab_size, (B, SRC)).to(device)
lang_ids = torch.randint(0, cfg.num_languages, (B, SRC)).to(device)
tgt_ids  = torch.randint(4, cfg.vocab_size, (B, TGT)).to(device)

# Add some padding
src_ids[0, 20:] = cfg.pad_id
src_ids[1, 22:] = cfg.pad_id
lang_ids[0, 20:] = 0
lang_ids[1, 22:] = 0

with torch.no_grad():
    logits = model(src_ids, lang_ids, tgt_ids)

print(f'[PASS] Forward pass: input [{B},{SRC}] → logits {logits.shape}')
assert logits.shape == (B, TGT, cfg.vocab_size), 'Shape mismatch!'

# Check no NaNs
assert not torch.isnan(logits).any(), 'NaN in logits!'
print(f'[PASS] No NaNs in output')

# Check loss computation
criterion = nn.CrossEntropyLoss(ignore_index=cfg.pad_id)
loss = criterion(logits.reshape(-1, cfg.vocab_size), tgt_ids.reshape(-1))
print(f'[PASS] Loss computation: {loss.item():.4f}  (random init, expected ~log({cfg.vocab_size})={math.log(cfg.vocab_size):.2f})')

# Check beam search (on 1 sample)
sample_src  = src_ids[:1]
sample_lang = lang_ids[:1]
output_ids  = beam_decoder.decode(sample_src, sample_lang, max_len=20)
print(f'[PASS] Beam search: output token ids = {output_ids[:10]}...')

# Check causal masking
tgt_check = torch.tensor([[cfg.bos_id, 100, 200, 300]]).to(device)
mask = make_tgt_mask(tgt_check, cfg.pad_id)
assert mask[0, 0, 0, 1] == False, 'Causal mask broken — future token visible!'
print(f'[PASS] Causal masking is correct')

# Language-ID embedding contribution check
tok_only  = torch.randint(4, cfg.vocab_size, (1, 8)).to(device)
lang_a    = torch.zeros(1, 8, dtype=torch.long).to(device)  # All English
lang_b    = torch.ones(1, 8, dtype=torch.long).to(device)   # All Hindi
emb_module = model.encoder.embedding
with torch.no_grad():
    out_a = emb_module(tok_only, lang_a)
    out_b = emb_module(tok_only, lang_b)
assert not torch.allclose(out_a, out_b), 'Language-ID embeddings have no effect!'
diff = (out_a - out_b).abs().mean().item()
print(f'[PASS] Language-ID embeddings have effect (mean diff = {diff:.4f})')

print('\nAll checks passed! Architecture is ready for Phase 3 (Training).')

## Cell 14 — Save Architecture to Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/CodeMixT'
os.makedirs(f'{DRIVE_DIR}/model', exist_ok=True)

# Save model architecture code as a standalone module
# (this cell writes the full model.py so Phase 3 training can import it)

MODEL_CODE = '''
# CodeMixT model module — auto-generated from Phase 2 notebook
# Import this in Phase 3 training notebook

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from typing import Optional

'''

# Save config
config_dict = {
    'vocab_size':      cfg.vocab_size,
    'pad_id':          cfg.pad_id,
    'bos_id':          cfg.bos_id,
    'eos_id':          cfg.eos_id,
    'num_languages':   cfg.num_languages,
    'lang_embed_dim':  cfg.lang_embed_dim,
    'd_model':         cfg.d_model,
    'd_ff':            cfg.d_ff,
    'num_heads':       cfg.num_heads,
    'num_enc_layers':  cfg.num_enc_layers,
    'num_dec_layers':  cfg.num_dec_layers,
    'max_seq_len':     cfg.max_seq_len,
    'dropout':         cfg.dropout,
    'beam_size':       cfg.beam_size,
    'max_gen_len':     cfg.max_gen_len,
}

import json
with open(f'{DRIVE_DIR}/model/config.json', 'w') as f:
    json.dump(config_dict, f, indent=2)

# Save initialized (untrained) model weights
torch.save(model.state_dict(), f'{DRIVE_DIR}/model/codemix_t_init.pt')

print('Saved to Google Drive:')
print(f'  {DRIVE_DIR}/model/config.json')
print(f'  {DRIVE_DIR}/model/codemix_t_init.pt')
print('\n=== PHASE 2 COMPLETE ===')
print('Next: Phase 3 — Training loop, LR scheduling, checkpointing')